# Bulk Data Coverage Analysis using AWS Athena

This notebook analyzes dataset coverage using AWS Athena to query parquet files in S3.

It creates temporary external tables and runs SQL queries to get coverage statistics.

**Key features:**
- Creates temporary Athena tables for parquet data
- Runs SQL queries for statistics (faster than reading files directly)
- Handles large datasets that don't fit in memory
- Automatic schema detection
- Cleanup of temporary tables

## Setup

In [ ]:
import boto3
import pandas as pd
import time
import uuid
from io import BytesIO
from typing import Optional, Dict, List

In [ ]:
# Configuration
BUCKET = "carc-ext-ant"
REGION = "us-east-1"
DATABASE = "default"

# S3 prefix containing parquet data
DATA_PREFIX = "20260108/Full/ant_permit_data"

print(f"Bucket: {BUCKET}")
print(f"Region: {REGION}")
print(f"Database: {DATABASE}")
print(f"Data prefix: {DATA_PREFIX}")

## 1. Athena Coverage Analyzer Class

This class handles all Athena operations for coverage analysis.

In [ ]:
class AthenaCoverageAnalyzer:
    """Analyze bulk data coverage using AWS Athena."""

    def __init__(
        self,
        bucket: str,
        region: str = "us-east-1",
        database: str = "default",
        output_location: Optional[str] = None,
    ):
        """
        Initialize the analyzer.

        Args:
            bucket: S3 bucket containing the bulk data
            region: AWS region
            database: Athena database to use
            output_location: S3 location for query results
        """
        self.bucket = bucket
        self.region = region
        self.database = database
        self.output_location = output_location or f"s3://{bucket}/athena-results/"

        self.s3 = boto3.client("s3", region_name=region)
        self.athena = boto3.client("athena", region_name=region)
        self.glue = boto3.client("glue", region_name=region)

        self._temp_tables = []

    def run_query(
        self,
        query: str,
        wait: bool = True,
        timeout: int = 300,
    ) -> Optional[pd.DataFrame]:
        """
        Run an Athena query and return results as DataFrame.

        Args:
            query: SQL query to run
            wait: Whether to wait for results
            timeout: Timeout in seconds

        Returns:
            DataFrame with results, or None if not waiting
        """
        response = self.athena.start_query_execution(
            QueryString=query,
            QueryExecutionContext={"Database": self.database},
            ResultConfiguration={"OutputLocation": self.output_location}
        )

        query_id = response["QueryExecutionId"]

        if not wait:
            return None

        # Wait for query to complete
        start_time = time.time()
        while time.time() - start_time < timeout:
            status = self.athena.get_query_execution(QueryExecutionId=query_id)
            state = status["QueryExecution"]["Status"]["State"]

            if state == "SUCCEEDED":
                # Get results
                results = self.athena.get_query_results(QueryExecutionId=query_id)

                # Parse results into DataFrame
                columns = [col["Name"] for col in results["ResultSet"]["ResultSetMetadata"]["ColumnInfo"]]
                rows = []
                for row in results["ResultSet"]["Rows"][1:]:  # Skip header
                    rows.append([cell.get("VarCharValue", "") for cell in row["Data"]])

                return pd.DataFrame(rows, columns=columns)

            elif state in ["FAILED", "CANCELLED"]:
                reason = status["QueryExecution"]["Status"].get("StateChangeReason", "Unknown")
                raise Exception(f"Query {state}: {reason}")

            time.sleep(2)

        raise TimeoutError(f"Query timed out after {timeout} seconds")

    def create_temp_table(self, s3_location: str, table_name: Optional[str] = None) -> str:
        """
        Create a temporary external table for parquet data.

        Args:
            s3_location: S3 location of the parquet data
            table_name: Optional table name (auto-generated if not provided)

        Returns:
            The created table name
        """
        if not table_name:
            table_name = f"temp_coverage_{uuid.uuid4().hex[:8]}"

        # Ensure s3_location ends with /
        if not s3_location.endswith("/"):
            s3_location += "/"

        create_sql = f"""
        CREATE EXTERNAL TABLE IF NOT EXISTS `{table_name}`
        STORED AS PARQUET
        LOCATION '{s3_location}'
        """

        try:
            self.run_query(create_sql)
            self._temp_tables.append(table_name)
            print(f"Created table: {table_name}")
            return table_name
        except Exception as e:
            print(f"Error creating table: {e}")
            # Try alternative approach with schema detection
            return self._create_table_with_schema(s3_location, table_name)

    def _create_table_with_schema(self, s3_location: str, table_name: str) -> str:
        """Create table by first detecting schema from a sample file."""
        import pyarrow.parquet as pq
        
        # Find a parquet file to get schema
        prefix = s3_location.replace(f"s3://{self.bucket}/", "")
        paginator = self.s3.get_paginator("list_objects_v2")

        parquet_key = None
        for page in paginator.paginate(Bucket=self.bucket, Prefix=prefix):
            for obj in page.get("Contents", []):
                if obj["Key"].endswith(".parquet") and obj["Size"] > 1000:
                    parquet_key = obj["Key"]
                    break
            if parquet_key:
                break

        if not parquet_key:
            raise ValueError(f"No parquet files found at {s3_location}")

        # Read only the schema from parquet (not the data)
        print(f"  Reading schema from: {parquet_key}")
        response = self.s3.get_object(Bucket=self.bucket, Key=parquet_key)
        parquet_file = pq.ParquetFile(BytesIO(response["Body"].read()))
        schema = parquet_file.schema_arrow

        # Map PyArrow types to Athena types
        def arrow_to_athena(arrow_type) -> str:
            type_str = str(arrow_type)
            if type_str.startswith("int64"):
                return "BIGINT"
            elif type_str.startswith("int32"):
                return "INT"
            elif type_str.startswith("float64") or type_str.startswith("double"):
                return "DOUBLE"
            elif type_str.startswith("float32") or type_str.startswith("float"):
                return "FLOAT"
            elif type_str.startswith("bool"):
                return "BOOLEAN"
            elif type_str.startswith("timestamp"):
                return "TIMESTAMP"
            elif type_str.startswith("date"):
                return "DATE"
            elif type_str.startswith("decimal"):
                return "DOUBLE"  # Simplify to DOUBLE for compatibility
            else:
                return "STRING"

        columns = []
        for field in schema:
            athena_type = arrow_to_athena(field.type)
            columns.append(f"`{field.name}` {athena_type}")

        create_sql = f"""
        CREATE EXTERNAL TABLE IF NOT EXISTS `{table_name}` (
          {', '.join(columns)}
        )
        STORED AS PARQUET
        LOCATION '{s3_location}'
        """

        self.run_query(create_sql)
        self._temp_tables.append(table_name)
        print(f"Created table with schema: {table_name}")
        print(f"  Columns: {len(columns)}")
        return table_name

    def get_row_count(self, table_name: str) -> int:
        """Get total row count."""
        df = self.run_query(f"SELECT COUNT(*) as cnt FROM {table_name}")
        return int(df["cnt"].iloc[0])

    def get_column_stats(self, table_name: str, column: str) -> Dict:
        """Get statistics for a specific column."""
        stats_query = f"""
        SELECT
            COUNT(*) as total_rows,
            COUNT({column}) as non_null_count,
            COUNT(DISTINCT {column}) as unique_count
        FROM {table_name}
        """
        df = self.run_query(stats_query)

        stats = {
            "total_rows": int(df["total_rows"].iloc[0]),
            "non_null_count": int(df["non_null_count"].iloc[0]),
            "unique_count": int(df["unique_count"].iloc[0]),
        }
        stats["null_pct"] = round((1 - stats["non_null_count"] / stats["total_rows"]) * 100, 2)

        return stats

    def get_numeric_stats(self, table_name: str, column: str) -> Dict:
        """Get statistics for a numeric column."""
        query = f"""
        SELECT
            MIN({column}) as min_val,
            MAX({column}) as max_val,
            AVG({column}) as avg_val,
            APPROX_PERCENTILE({column}, 0.5) as median_val
        FROM {table_name}
        WHERE {column} IS NOT NULL
        """
        df = self.run_query(query)

        return {
            "min": float(df["min_val"].iloc[0]) if df["min_val"].iloc[0] else None,
            "max": float(df["max_val"].iloc[0]) if df["max_val"].iloc[0] else None,
            "mean": float(df["avg_val"].iloc[0]) if df["avg_val"].iloc[0] else None,
            "median": float(df["median_val"].iloc[0]) if df["median_val"].iloc[0] else None,
        }

    def get_value_distribution(self, table_name: str, column: str, limit: int = 10) -> pd.DataFrame:
        """Get value distribution for a column."""
        query = f"""
        SELECT {column}, COUNT(*) as count
        FROM {table_name}
        GROUP BY {column}
        ORDER BY count DESC
        LIMIT {limit}
        """
        return self.run_query(query)

    def get_geographic_coverage(self, table_name: str) -> Dict:
        """Get geographic coverage statistics."""
        result = {}

        # Check for state column
        state_query = f"""
        SELECT state, COUNT(*) as count
        FROM {table_name}
        WHERE state IS NOT NULL
        GROUP BY state
        ORDER BY count DESC
        LIMIT 20
        """
        try:
            df = self.run_query(state_query)
            result["states"] = df.to_dict("records")
            result["state_count"] = len(df)
        except:
            pass

        # Check for zip_code column
        zip_query = f"""
        SELECT COUNT(DISTINCT zip_code) as unique_zips
        FROM {table_name}
        WHERE zip_code IS NOT NULL
        """
        try:
            df = self.run_query(zip_query)
            result["unique_zip_codes"] = int(df["unique_zips"].iloc[0])
        except:
            pass

        return result

    def get_full_coverage_report(self, table_name: str) -> Dict:
        """Generate a comprehensive coverage report."""
        print("Generating coverage report...")

        report = {
            "table_name": table_name,
            "row_count": self.get_row_count(table_name),
        }

        # Get column info
        describe_query = f"DESCRIBE {table_name}"
        columns_df = self.run_query(describe_query)
        report["columns"] = columns_df.to_dict("records")

        # Get geographic coverage
        print("  Analyzing geographic coverage...")
        report["geographic"] = self.get_geographic_coverage(table_name)

        # Get category distributions
        print("  Analyzing category distributions...")
        category_columns = ["permit_classifier_name", "building_permit_status_name"]
        report["categories"] = {}
        for col in category_columns:
            try:
                df = self.get_value_distribution(table_name, col)
                report["categories"][col] = df.to_dict("records")
            except:
                pass

        # Get numeric stats
        print("  Analyzing numeric fields...")
        numeric_columns = ["permit_job_value", "permit_fees"]
        report["numeric_stats"] = {}
        for col in numeric_columns:
            try:
                report["numeric_stats"][col] = self.get_numeric_stats(table_name, col)
            except:
                pass

        return report

    def print_report(self, report: Dict):
        """Print a formatted coverage report."""
        print("\n" + "=" * 70)
        print("ATHENA COVERAGE REPORT")
        print("=" * 70)

        print(f"\nTable: {report['table_name']}")
        print(f"Total rows: {report['row_count']:,}")
        print(f"Columns: {len(report['columns'])}")

        print("\n" + "-" * 70)
        print("SCHEMA")
        print("-" * 70)
        for col in report["columns"]:
            print(f"  {col.get('col_name', col.get('column_name', 'unknown')):30} {col.get('data_type', 'unknown')}")

        if report.get("geographic"):
            print("\n" + "-" * 70)
            print("GEOGRAPHIC COVERAGE")
            print("-" * 70)
            geo = report["geographic"]
            if "state_count" in geo:
                print(f"States: {geo['state_count']}")
            if "unique_zip_codes" in geo:
                print(f"Zip codes: {geo['unique_zip_codes']:,}")
            if "states" in geo:
                print("\nTop states:")
                for s in geo["states"][:10]:
                    print(f"  {s['state']}: {int(s['count']):,}")

        if report.get("categories"):
            print("\n" + "-" * 70)
            print("CATEGORY DISTRIBUTIONS")
            print("-" * 70)
            for col, values in report["categories"].items():
                print(f"\n{col}:")
                for v in values[:10]:
                    col_val = list(v.values())[0]
                    count = int(list(v.values())[1])
                    print(f"  {col_val}: {count:,}")

        if report.get("numeric_stats"):
            print("\n" + "-" * 70)
            print("NUMERIC STATISTICS")
            print("-" * 70)
            for col, stats in report["numeric_stats"].items():
                print(f"\n{col}:")
                if stats.get("min") is not None:
                    print(f"  Min: {stats['min']:,.2f}")
                if stats.get("max") is not None:
                    print(f"  Max: {stats['max']:,.2f}")
                if stats.get("mean") is not None:
                    print(f"  Mean: {stats['mean']:,.2f}")
                if stats.get("median") is not None:
                    print(f"  Median: {stats['median']:,.2f}")

    def cleanup(self):
        """Drop temporary tables."""
        for table in self._temp_tables:
            try:
                self.run_query(f"DROP TABLE IF EXISTS {table}")
                print(f"Dropped table: {table}")
            except:
                pass
        self._temp_tables = []

## 2. Initialize the Analyzer

Create an instance of the analyzer with your bucket and configuration.

In [ ]:
analyzer = AthenaCoverageAnalyzer(
    bucket=BUCKET,
    region=REGION,
    database=DATABASE,
)

print(f"Analyzer initialized")
print(f"  Output location: {analyzer.output_location}")

## 3. Create Temporary Table

Create a temporary external table pointing to the parquet data.

In [ ]:
s3_location = f"s3://{BUCKET}/{DATA_PREFIX}"
print(f"Creating table for: {s3_location}")

table_name = analyzer.create_temp_table(s3_location)

## 4. Basic Statistics

Get row count and column information.

In [ ]:
# Get total row count
row_count = analyzer.get_row_count(table_name)
print(f"Total rows: {row_count:,}")

In [ ]:
# Preview sample data
sample_df = analyzer.run_query(f"SELECT * FROM {table_name} LIMIT 10")
sample_df

## 5. Column Statistics

Get detailed statistics for specific columns.

In [ ]:
# Get statistics for the state column
state_stats = analyzer.get_column_stats(table_name, "state")
print("State column statistics:")
print(f"  Total rows: {state_stats['total_rows']:,}")
print(f"  Non-null count: {state_stats['non_null_count']:,}")
print(f"  Unique values: {state_stats['unique_count']:,}")
print(f"  Null %: {state_stats['null_pct']}%")

In [ ]:
# Get statistics for zip_code column
zip_stats = analyzer.get_column_stats(table_name, "zip_code")
print("Zip code column statistics:")
print(f"  Total rows: {zip_stats['total_rows']:,}")
print(f"  Non-null count: {zip_stats['non_null_count']:,}")
print(f"  Unique values: {zip_stats['unique_count']:,}")
print(f"  Null %: {zip_stats['null_pct']}%")

## 6. Geographic Coverage

Analyze geographic distribution of the data.

In [ ]:
# Get geographic coverage
geo_coverage = analyzer.get_geographic_coverage(table_name)

print("Geographic Coverage:")
print("=" * 60)
if "state_count" in geo_coverage:
    print(f"Total states: {geo_coverage['state_count']}")
if "unique_zip_codes" in geo_coverage:
    print(f"Unique zip codes: {geo_coverage['unique_zip_codes']:,}")

In [ ]:
# Show state distribution
state_dist = analyzer.get_value_distribution(table_name, "state", limit=20)
print("State Distribution:")
state_dist

## 7. Category Distributions

Analyze distribution of categorical fields.

In [ ]:
# Permit type distribution
try:
    permit_types = analyzer.get_value_distribution(table_name, "permit_classifier_name")
    print("Permit Type Distribution:")
    display(permit_types)
except Exception as e:
    print(f"Could not get permit type distribution: {e}")

In [ ]:
# Permit status distribution
try:
    permit_status = analyzer.get_value_distribution(table_name, "building_permit_status_name")
    print("Permit Status Distribution:")
    display(permit_status)
except Exception as e:
    print(f"Could not get permit status distribution: {e}")

## 8. Numeric Statistics

Analyze numeric fields like job values and fees.

In [ ]:
# Permit fees statistics
try:
    fees_stats = analyzer.get_numeric_stats(table_name, "permit_fees")
    print("Permit Fees Statistics:")
    print("=" * 40)
    if fees_stats.get("min") is not None:
        print(f"  Min: ${fees_stats['min']:,.2f}")
    if fees_stats.get("max") is not None:
        print(f"  Max: ${fees_stats['max']:,.2f}")
    if fees_stats.get("mean") is not None:
        print(f"  Mean: ${fees_stats['mean']:,.2f}")
    if fees_stats.get("median") is not None:
        print(f"  Median: ${fees_stats['median']:,.2f}")
except Exception as e:
    print(f"Could not get permit fees stats: {e}")

In [ ]:
# Example: High-value permits
high_value_query = f"""
SELECT 
    state,
    COUNT(*) as permit_count,
    AVG(permit_job_value) as avg_value,
    MAX(permit_job_value) as max_value
FROM {table_name}
WHERE permit_job_value > 100000
GROUP BY state
ORDER BY permit_count DESC
LIMIT 10
"""

try:
    high_value_stats = analyzer.run_query(high_value_query)
    print("High-Value Permits (>$100K) by State:")
    display(high_value_stats)
except Exception as e:
    print(f"Query failed: {e}")

## Cleanup

Drop the temporary table when done.

In [ ]:
# Cleanup temporary tables
analyzer.cleanup()